# 03_train_3d — Launcher Colab (A100)

Unico punto di accesso per eseguire il training 3D BraTS-PEDs su Google Colab.
Segue esattamente `step.md` (Fase 2): monta Drive, clona il codice da GitHub,
porta i dati e i pesi pre-addestrati in locale su `/content/`, poi lancia
`run_pipeline_3d.py`.

**Prerequisiti (Fase 1, gia' completata):** i file `train_3d.zip`, `val_3d.zip`,
`test_3d.zip` e `split_3d.json` sono gia' presenti su Google Drive in
`MyDrive/BraTS_Project/data/`, e i pesi pre-addestrati (`model_swinvit.pt`,
`model_best_fold_0.pth` per SwinUNETR; `model.pt` per SegResNet — bundle MONAI
Model Zoo `brats_mri_segmentation`, `init_filters=16`) sono in
`MyDrive/BraTS_Project/pretrained/`. Il progetto e' 100% transfer learning: per
allenare SegResNet serve `model.pt` caricato su Drive in questa cartella prima
di eseguire il notebook.

**Runtime:** verificare `Runtime > Change runtime type > A100 GPU` prima di
eseguire la prima cella.

**IMPORTANTE — prima di eseguire questo notebook:** la Sezione 2 clona il codice da GitHub (branch `estensione-pipeline-3d`). Se hai appena modificato in locale `run_pipeline_3d.py` / `src/dataset_3d.py` / `src/losses_monai.py` (fix classe CC: pesi per-classe + patch sampling bilanciato, `DEFAULT_CLASS_SAMPLE_RATIOS`), **committa e pusha quelle modifiche prima di lanciare il notebook** — altrimenti Colab clona la versione precedente, senza il fix.

**Fix classe CC (Cystic Component, la sub-regione piu' rara — vedi EDA, `scripts/compute_class_freq.py`):** il ramo `--loss=dice_focal` ora legge automaticamente `region_class_weights.json` da `--data-root` (Sezione 3 lo copia li'), e il patch sampling di training (`RandCropByLabelClassesd`) sovra-pesa di default CC 2x rispetto alle altre sub-regioni. Il ramo `--loss=gsl` resta invariato (baseline di confronto).

**`region_class_weights.json` viene calcolato QUI, non caricato da Drive:** la Sezione 3, subito dopo aver decompresso i dati, scansiona `data/processed_3d/train/` (SOLO i soggetti train, per costruzione — nessun rischio di leakage da val/test) e scrive i pesi direttamente in `data/processed_3d/region_class_weights.json`. Niente file da preparare in anticipo ne' da caricare a mano.

## 1. Mount Google Drive

Necessario per accedere ai dati (zip) e ai pesi pre-addestrati caricati
manualmente nella Fase 1. Autorizzare l'accesso quando richiesto dal popup.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

## 2. Setup Ambiente & Clone

Installa le dipendenze minime (`monai`, `nibabel`, `tqdm` — coerenti con
`requirements.txt`; `torch` e' gia' preinstallato con build CUDA nell'immagine
Colab standard, quindi non va reinstallato) e clona **esclusivamente** il
branch `estensione-pipeline-3d` del repository ufficiale, che contiene tutto
il codice sorgente (`src/`, `run_pipeline_3d.py`) gia' scritto e testato in
locale. Il clone NON porta dati ne' pesi: sono esclusi da `.gitignore` e
vengono recuperati da Drive nelle celle successive.

In [ ]:
!pip install -q monai nibabel tqdm

!git clone -b estensione-pipeline-3d https://github.com/Drastid/BraTS-PEDs-brain-tumour-segmentation.git BraTS-PEDs-3D

%cd BraTS-PEDs-3D

## 3. I/O — Dati

Crea `data/processed_3d/`, copia i tre archivi zip (train/val/test) e
`split_3d.json` da Drive allo storage locale della VM Colab (`/content/`,
molto piu' veloce di Drive montato via FUSE per le letture ripetute che MONAI
fara' durante il training), li decomprime silenziosamente e posiziona
`split_3d.json` nel punto atteso dal codice (`data/split_3d.json`).

Calcola inoltre `region_class_weights.json` (pesi per-classe, Eq. 13 — vedi `src/class_weights.py`) direttamente su `data/processed_3d/train/`, cioe' SOLO sui soggetti train gia' decompressi qui sopra: nessun file da preparare in anticipo, nessun rischio di calcolare i pesi anche su val/test. Il file va a `data/processed_3d/region_class_weights.json`, dove il ramo `--loss=dice_focal` lo cerca gia' (stessa convenzione di `gsl_class_weights.json`, che invece va ancora aggiunto a mano in `MyDrive/BraTS_Project/data/` se un domani vuoi pesare anche la GSL).

In [ ]:
import os
import shutil

DRIVE_DATA = "/content/drive/MyDrive/BraTS_Project/data"

os.makedirs("data/processed_3d", exist_ok=True)

# Copia gli zip da Drive allo storage locale di Colab
for fname in ["train_3d.zip", "val_3d.zip", "test_3d.zip"]:
    shutil.copy(os.path.join(DRIVE_DATA, fname), os.path.join("data", fname))

# Decompressione silenziosa (-q) di ciascuno split dentro data/processed_3d/
!unzip -q data/train_3d.zip -d data/processed_3d/
!unzip -q data/val_3d.zip -d data/processed_3d/
!unzip -q data/test_3d.zip -d data/processed_3d/

# split_3d.json va copiato (non zippato) direttamente in data/
shutil.copy(os.path.join(DRIVE_DATA, "split_3d.json"), "data/split_3d.json")

# Pesi per-classe per il ramo --loss=dice_focal (fix classe CC), calcolati QUI
# su data/processed_3d/train/ — SOLO soggetti train (quella cartella li contiene
# per costruzione, nessun soggetto val/test), quindi nessun rischio di leakage.
# Stessa formula (Eq.13) gia' usata per la GSL, vedi src/class_weights.py.
import json as _json
from src.class_weights import count_voxels_from_dir, weights_from_counts

_counts = count_voxels_from_dir("data/processed_3d/train", num_classes=5)
_freq = _counts / _counts.sum()
_weights = weights_from_counts(_counts)

print("--- Frequenze voxel per classe (TRAIN ONLY) ---")
for _name, _c, _f, _w in zip(("BG", "ET", "NET", "CC", "ED"), _counts, _freq, _weights):
    print(f"  {_name:4s}  voxel={int(_c):>12d}  freq={100*_f:7.4f}%  weight={_w:.4f}")

with open("data/processed_3d/region_class_weights.json", "w") as f:
    _json.dump({"weights": _weights.tolist()}, f, indent=2)
print("region_class_weights.json scritto in data/processed_3d/ (fix classe CC attivo, train-only).")

print("train:", len(os.listdir("data/processed_3d/train")))
print("val  :", len(os.listdir("data/processed_3d/val")))
print("test :", len(os.listdir("data/processed_3d/test")))

## 4. Caricamento Pesi Pre-addestrati

**Il progetto 3D e' 100% transfer learning** (decisione utente): entrambe le
architetture (SegResNet, SwinUNETR) partono SEMPRE da pesi pre-addestrati, mai
da zero. I pesi NON vengono scaricati da internet a runtime: sono gia' stati
caricati manualmente su Drive (Fase 1) e vengono qui copiati fisicamente nella
cartella locale `weights/` del progetto Colab:

- `model_swinvit.pt` e/o `model_best_fold_0.pth` — checkpoint SwinUNETR (SSL
  NVIDIA o HuggingFace/BrainSegFounder).
- `model.pt` — checkpoint SegResNet (bundle MONAI Model Zoo
  `brats_mri_segmentation`, `init_filters=16`, verificato: 81/83 tensori del
  backbone caricano correttamente, solo la head 3→5 classi resta esclusa).

I flag pesi sono **specifici per architettura** (`--pretrained-swinunetr`,
`--pretrained-segresnet`): un checkpoint SwinUNETR non ha alcuna chiave in
comune con SegResNet, quindi non esiste un `--weights-path` condiviso. Con
`--pretrained=auto` (default), l'assenza del checkpoint per un'architettura
richiesta blocca l'esecuzione PRIMA del training (policy transfer learning
enforced in `run_pipeline_3d.py`).

In [ ]:
DRIVE_PRETRAINED = "/content/drive/MyDrive/BraTS_Project/pretrained"

os.makedirs("weights", exist_ok=True)

for fname in ["model_swinvit.pt", "model_best_fold_0.pth", "model.pt"]:
    shutil.copy(os.path.join(DRIVE_PRETRAINED, fname), os.path.join("weights", fname))

print(os.listdir("weights"))

## 5. Training

Lancia la pipeline di training via CLI (`run_pipeline_3d.py`). La barra di
progresso per epoca (tempo stimato, loss corrente) e' gestita **internamente**
dal training loop in `src/train_3d.py`, che avvolge gia' i batch del
`DataLoader` con `tqdm` (`train_one_epoch_3d`, `train_one_epoch_gsl_3d`,
`evaluate_3d`) — non serve alcuna configurazione aggiuntiva in questa cella
per vederla: appare automaticamente nell'output sotto.

`--early-stopping` e' **attivo** (patience=15, min-delta=1e-4): il training si
ferma prima di `--epochs 100` se `dice_mean_fg` di validazione non migliora per
15 epoche consecutive, risparmiando ore-GPU su un plateau. La selezione di
`best.pth` resta comunque sempre legata al miglior `dice_mean_fg` grezzo visto
(indipendente dallo stop). `history.json` traccia, per ogni epoca: train loss,
Dice/HD95 di validazione per-classe (ET/NET/CC/ED) + medie, LR backbone/head,
stato di freeze — sufficiente per valutare l'andamento del training a posteriori.

Esempio: baseline DiceFocalLoss su SwinUNETR (`--arch swinunetr`) con i pesi
SSL NVIDIA (`model_swinvit.pt`) caricati dalla cella precedente
(`--pretrained-swinunetr`). Per allenare **entrambe** le architetture in
sequenza, usare la cella "5b" subito sotto (`--all`) invece di questa: pesca
automaticamente `--pretrained-segresnet`/`--pretrained-swinunetr` per l'arch
corrente del ciclo, e con `--pretrained=auto` entrambi sono obbligatori (nessun
training da zero per errore).

In [ ]:
!python run_pipeline_3d.py \
    --arch swinunetr \
    --loss dice_focal \
    --data-root data/processed_3d \
    --roi 128 128 128 \
    --batch-size 2 \
    --num-samples 2 \
    --num-workers 4 \
    --epochs 100 \
    --pretrained auto \
    --pretrained-swinunetr weights/model_swinvit.pt \
    --early-stopping \
    --es-patience 15 \
    --es-min-delta 1e-4 \
    --run-name run01 \
    --backup-dir /content/drive/MyDrive/BraTS_Project/checkpoints

### 5b. (Alternativa) Allenare ENTRAMBE le architetture in sequenza con `--all`

Sostituisce la cella precedente: allena SegResNet e poi SwinUNETR in sequenza,
ciascuna nella propria sottocartella `<run-name>/<arch>_<loss>/`. Con
`--pretrained=auto` (default) **entrambi** i flag pesi sono obbligatori — se ne
manca uno, lo script si ferma PRIMA di allenare qualsiasi modello (nessun
training da zero per errore, policy transfer learning enforced in
`_validate_pretrained_weights`). Un fallimento (es. OOM) su un'architettura non
cancella i risultati dell'altra già completata.

`--early-stopping` e' attivo (patience=15) per **ciascuna** architettura
indipendentemente: SegResNet e SwinUNETR possono fermarsi a epoche diverse a
seconda di quando il loro `dice_mean_fg` smette di migliorare.

In [ ]:
!python run_pipeline_3d.py \
    --all \
    --loss dice_focal \
    --data-root data/processed_3d \
    --roi 128 128 128 \
    --batch-size 2 \
    --num-samples 2 \
    --num-workers 4 \
    --epochs 100 \
    --pretrained auto \
    --pretrained-segresnet weights/model.pt \
    --pretrained-swinunetr weights/model_swinvit.pt \
    --early-stopping \
    --es-patience 15 \
    --es-min-delta 1e-4 \
    --run-name run01 \
    --backup-dir /content/drive/MyDrive/BraTS_Project/checkpoints

### 5c. Ensemble SegResNet + SwinUNETR

SegResNet (CNN) e SwinUNETR (transformer) hanno bias induttivi diversi — i loro errori sono parzialmente decorrelati, la condizione che rende utile un ensemble. Questa cella NON allena nulla: carica i due `best.pth` gia' prodotti dalla Sezione 5b, media le loro probabilita' softmax voxel-per-voxel (`run_ensemble_3d.py` — vedi `src/ensemble_3d.py`), valuta l'ensemble sul test set, poi confronta a TRE vie (SegResNet, SwinUNETR, Ensemble) con un test di Wilcoxon sui Dice per-soggetto e stampa un verdetto: l'ensemble e' accettato solo se `dice_mean_fg` migliora E la differenza e' statisticamente significativa (p<0.05).

Richiede che entrambe le architetture siano gia' state allenate E valutate (`--evaluate`, default ON) con lo STESSO `--run-name` e `--loss` (qui `run01` / `dice_focal` — perimetro deciso: l'ensemble su `--loss gsl` e' rimandato a dopo aver validato questo).

In [ ]:
!python run_ensemble_3d.py \
    --data-root data/processed_3d \
    --run-name run01 \
    --loss dice_focal \
    --segresnet-ckpt checkpoints/run01/segresnet_dice_focal/best.pth \
    --swinunetr-ckpt checkpoints/run01/swinunetr_dice_focal/best.pth \
    --roi 128 128 128

## 6. Risultati — Dice/HD95 per-classe (test set)

`run_pipeline_3d.py` esegue GIA' la valutazione 3D nativa sul test set subito dopo il training (`--evaluate` di default: ON) — questo notebook e' quindi gia' un unico entry point train+eval, non serve lanciare nient'altro. Questa cella legge `test_3d_metrics.json` (scritto in `evaluation_outputs/<run-name>/<arch>_<loss>/`, sia in locale sia — se `--backup-dir` era impostato — su Drive) e stampa il summary per-classe, con CC evidenziata: e' il numero da confrontare per capire se il fix (pesi + patch sampling) ha funzionato.

In [ ]:
import json

RUN_NAME = "run01"  # deve combaciare con --run-name usato nelle celle di training sopra
LOSS = "dice_focal"  # combacia con --loss usato per il training
FG_CLASSES = ("ET", "NET", "CC", "ED")

for arch in ("segresnet", "swinunetr"):
    metrics_path = f"evaluation_outputs/{RUN_NAME}/{arch}_{LOSS}/test_3d_metrics.json"
    if not __import__("os").path.isfile(metrics_path):
        continue  # arch non allenata in questo run (es. solo --arch segresnet)

    with open(metrics_path) as f:
        summary = json.load(f)["summary"]

    print(f"\n=== {arch} / {LOSS} — test set (n soggetti: vedi per_subject) ===")
    for cls in FG_CLASSES:
        d_mean, d_std = summary.get(f"dice_{cls}_mean", float("nan")), summary.get(f"dice_{cls}_std", float("nan"))
        h_mean, h_std = summary.get(f"hd95_{cls}_mean", float("nan")), summary.get(f"hd95_{cls}_std", float("nan"))
        marker = "  <-- CC" if cls == "CC" else ""
        print(f"  {cls:4s}  dice={d_mean:.4f}±{d_std:.4f}   hd95={h_mean:6.2f}±{h_std:5.2f}{marker}")
    print(f"  {'mean_fg':4s}  dice={summary.get('dice_mean_fg_mean', float('nan')):.4f}±"
          f"{summary.get('dice_mean_fg_std', float('nan')):.4f}   "
          f"hd95={summary.get('hd95_mean_fg_mean', float('nan')):6.2f}±"
          f"{summary.get('hd95_mean_fg_std', float('nan')):5.2f}")

# L'ensemble (Sezione 5c) salva in una cartella "ensemble" a parte (non
# "<arch>_<loss>", perche' non e' una singola arch) — run_ensemble_3d.py
# stampa gia' il confronto a 3 vie + verdetto Wilcoxon alla fine della run;
# questo blocco lo ri-mostra qui per comodita' senza dover rilanciarlo.
ensemble_metrics_path = f"evaluation_outputs/{RUN_NAME}/ensemble/test_3d_metrics.json"
if __import__("os").path.isfile(ensemble_metrics_path):
    with open(ensemble_metrics_path) as f:
        ens_summary = json.load(f)["summary"]
    print(f"\n=== ensemble (segresnet+swinunetr) / {LOSS} — test set ===")
    for cls in FG_CLASSES:
        d_mean, d_std = ens_summary.get(f"dice_{cls}_mean", float("nan")), ens_summary.get(f"dice_{cls}_std", float("nan"))
        h_mean, h_std = ens_summary.get(f"hd95_{cls}_mean", float("nan")), ens_summary.get(f"hd95_{cls}_std", float("nan"))
        marker = "  <-- CC" if cls == "CC" else ""
        print(f"  {cls:4s}  dice={d_mean:.4f}±{d_std:.4f}   hd95={h_mean:6.2f}±{h_std:5.2f}{marker}")
    print(f"  {'mean_fg':4s}  dice={ens_summary.get('dice_mean_fg_mean', float('nan')):.4f}±"
          f"{ens_summary.get('dice_mean_fg_std', float('nan')):.4f}   "
          f"hd95={ens_summary.get('hd95_mean_fg_mean', float('nan')):6.2f}±"
          f"{ens_summary.get('hd95_mean_fg_std', float('nan')):5.2f}")

    verdict_path = f"evaluation_outputs/{RUN_NAME}/ensemble/ensemble_comparison.json"
    if __import__("os").path.isfile(verdict_path):
        with open(verdict_path) as f:
            verdict = json.load(f)["verdict"]
        stato = "ACCETTATO" if verdict["accepted"] else "NON accettato"
        print(f"  Verdetto: {stato}  (improved={verdict['improved']}, "
              f"significant={verdict['significant']}, p={verdict['p_value']:.4f})")